In [35]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os


In [36]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [37]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)


In [38]:
# create state 

class LLMState(TypedDict):

    question : str
    answer: str

In [39]:
def llm_qa(state: LLMState) -> LLMState:
    
    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question \n {question}'

    # ask that question to llm
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state



In [40]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa',llm_qa)

# add edges
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# compile the graph
workflow = graph.compile()

In [41]:
# execute the graph
initial_state = {'question':'What is the capital of pakistan?'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'What is the capital of pakistan?', 'answer': 'The capital of Pakistan is **Islamabad**.'}
